# LAB 06 — Delta Lake Optimization: From Fragmentation to Performance

> **Ticket DATA-163** · From: Finance · Priority: High
>
> The finance dashboard reads the orders table, and over the last months it has become slow. The ETL job appends small batches to the table many times a day, so the table now has many small files. Find the cause in the table metadata and compact the table. Then measure data skipping to choose a data layout for the product filter of the dashboard. Finally, set the table up so that it stays fast without manual tuning.

**Definition of done**
- Task 1: you know the file count, the size and the recent operations of `bronze.orders_lab`. You read them from the table metadata.
- Task 2: `bronze.orders_lab` has fewer files after compaction, and its history shows the compaction. Stretch (Tasks 2b–2c): the history also shows `VACUUM`.
- Task 3: `bronze.orders_lab_zorder` is Z-ordered by `product_id`, and `bronze.orders_lab_liquid` has `clusteringColumns = ['product_id']`. The benchmark shows how many files hold one product in each table.
- Task 4 (stretch): automatic liquid clustering is on for `bronze.orders_lab_auto`.

A task is done when its validation cell prints `Task N OK`. For Tasks 2–4, **Check your work** in the app must also pass. Task 1 only reads metadata, so it has no Check. Tasks 2b, 2c and 4 are stretch tasks. Do them if you finish early, or after the course. No later task needs them.

Work in the Workshop Guide app in the left window, and paste the code into the marked cells of this notebook.

**Prerequisites:** 06 — Delta Optimization demo

## Setup

In [ ]:
%run ../../setup/00_setup

In [ ]:
# Define the names of the four lab tables
TABLE_BASE   = f"{CATALOG}.{BRONZE_SCHEMA}.orders_lab"
TABLE_ZORDER = f"{CATALOG}.{BRONZE_SCHEMA}.orders_lab_zorder"
TABLE_LIQUID = f"{CATALOG}.{BRONZE_SCHEMA}.orders_lab_liquid"
TABLE_AUTO   = f"{CATALOG}.{BRONZE_SCHEMA}.orders_lab_auto"

print(f"Base table    : {TABLE_BASE}")
print(f"Z-ORDER table : {TABLE_ZORDER}")
print(f"Liquid table  : {TABLE_LIQUID}")
print(f"Auto table    : {TABLE_AUTO}")

## Preparation — Creating a Fragmented Delta Table

ETL jobs that write often, such as streaming or frequent micro-batch jobs, add a few small files with every batch. After months of such writes the table has hundreds of small Parquet files, and many small files make every scan of the table slower.

The next cell builds such a table. It writes 2 million synthetic orders as 200 files. Then it appends 20 micro-batches of about 20 000 rows, with 5 files each. The result is about 2.4 million rows in about 300 small files.

The cell is provided. Run it as it is, without changes.

In [ ]:
# PROVIDED (run as it is): build the fragmented table orders_lab
from pyspark.sql import functions as F

# Generate 2 million synthetic orders
# - 1 000 distinct product_ids  (PROD_0001 … PROD_1000)
# - 50 000 distinct customer_ids
# - 50 distinct store_ids
# Used by every task; Task 3 compares data skipping on product_id.
N = 2_000_000

df_synthetic = (
    spark.range(N)
    .select(
        F.concat(F.lit("ORD_"), F.lpad(F.col("id").cast("string"), 8, "0"))
         .alias("order_id"),
        F.concat(
            F.lit("CUST_"),
            F.lpad((F.round(F.rand() * 49999).cast("long") + 1).cast("string"), 5, "0")
        ).alias("customer_id"),
        F.concat(
            F.lit("PROD_"),
            F.lpad((F.round(F.rand() * 999).cast("long") + 1).cast("string"), 4, "0")
        ).alias("product_id"),
        F.concat(F.lit("STORE_"), (F.round(F.rand() * 49).cast("long") + 1).cast("string"))
         .alias("store_id"),
        F.timestamp_seconds(
            F.unix_timestamp(F.lit("2023-01-01"), "yyyy-MM-dd") + (F.rand() * 365 * 86400).cast("long")
        ).alias("order_datetime"),
        (F.round(F.rand() * 9).cast("int") + 1).alias("quantity"),
        F.round(F.rand() * 490 + 10, 2).alias("unit_price"),
        F.round(F.rand() * 30).cast("int").alias("discount_percent"),
        F.element_at(
            F.array(F.lit("credit_card"), F.lit("debit_card"),
                    F.lit("paypal"), F.lit("bank_transfer"), F.lit("gift_card")),
            (F.round(F.rand() * 4).cast("int") + 1)
        ).alias("payment_method"),
    )
    # total_amount follows from the other columns: quantity x unit_price, minus the discount
    .withColumn("total_amount", F.round(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_percent") / 100), 2))
    .select("order_id", "customer_id", "product_id", "store_id", "order_datetime",
            "quantity", "unit_price", "discount_percent", "total_amount", "payment_method")
)

# Initial write: 200 files (high fragmentation)
spark.sql(f"DROP TABLE IF EXISTS {TABLE_BASE}")
(
    df_synthetic.repartition(200)
    .write
    .option("delta.autoOptimize.optimizeWrite", "false")
    .option("delta.autoOptimize.autoCompact", "false")
    .mode("overwrite")
    .saveAsTable(TABLE_BASE)
)
# delta.targetFileSize = 4mb: LAB ONLY. The table is small (under 60 MB), and OPTIMIZE targets ~1 GB files by default,
# so it would compact everything into ONE file and there would be nothing to skip in Task 3.
# With a 4 MB target the table keeps about a dozen files, so data skipping (Z-ORDER / Liquid) becomes visible.
# DEEP CLONE copies table properties, so the Task 3 copies inherit it.
spark.sql(f"""ALTER TABLE {TABLE_BASE} SET TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'false',
    'delta.autoOptimize.autoCompact'   = 'false',
    'delta.targetFileSize'             = '4mb'
)""")

# Simulate 20 micro-batch appends — 5 small files each = 100 more files
for i in range(20):
    (df_synthetic
     .sample(fraction=0.01, seed=i)  # ~20 000 rows per batch
     .repartition(5)
     .write.mode("append").saveAsTable(TABLE_BASE))

total_rows  = spark.table(TABLE_BASE).count()
total_files = spark.sql(f"DESCRIBE DETAIL {TABLE_BASE}").first()["numFiles"]
print(f"Fragmented table ready.")
print(f"  Total rows  : {total_rows:,}")
print(f"  Total files : {total_files}   <-- data is highly fragmented")

In [ ]:
# Show the schema and three sample rows (Task 3 needs the column names)
df_base = spark.table(TABLE_BASE)
df_base.printSchema()
display(df_base.limit(3))

## Task 1 — Read the table metadata

Before you change anything, find out what the dashboard has to read: how many files the table has, how big it is, and which operations wrote it. You start the same way at work whenever a report on a Delta table gets slow. One command returns one row of storage metrics. Another command returns one row per table version. Both read only the Delta log, not the data, so they are cheap on a table of any size.

Task 1 has two cells: **Task 1a** reads the details and **Task 1b** reads the history. The code for both cells is in the Workshop Guide.

In [ ]:
# Task 1a: paste the code from the Workshop Guide here (SQL or Python)

In [ ]:
# Task 1b: paste the code from the Workshop Guide here (SQL or Python)

In [ ]:
# -- Validation --
detail      = spark.sql(f"DESCRIBE DETAIL {TABLE_BASE}").first()
history_ops = [r["operation"] for r in spark.sql(f"DESCRIBE HISTORY {TABLE_BASE} LIMIT 5").collect()]
assert detail["format"] == "delta", "Table must be Delta format"
assert detail["numFiles"] > 100, \
    f"Expected a heavily fragmented table (>100 files), got {detail['numFiles']} — re-run the Preparation cell"
assert "WRITE" in history_ops, \
    f"Recent history should contain the micro-batch WRITE operations, got: {history_ops}"
print(f"Task 1 OK")
print(f"  Files      : {detail['numFiles']}")
print(f"  Size       : {detail['sizeInBytes']:,} bytes")
print(f"  Recent operations : {history_ops}")

## Task 2 — Compact the table and clean up the old files

Compaction merges small files into larger ones and commits the result as a new table version. You run it on tables that get many small writes. The files it replaced stay in storage for time travel until `VACUUM` deletes them.

In the Workshop Guide you first put the maintenance commands in their production order (**Task 2 plan**, in the Polish app **Plan Zadania 2**). Then you run them here in three cells:

- Task 2a compacts the table and reads the new file count.
- Task 2b lowers the retention of this table (lab only), then lists the files that `VACUUM` would delete, without deleting them.
- Task 2c runs `VACUUM`.

> Stretch: Tasks 2b and 2c are optional in class. Do them if you finish early, or after the course. Task 3 and the benchmark do not need them. Tasks 2a and 2c each have their own validation cell.

> The zero retention is for this lab only. It lets `VACUUM` delete the files that compaction has just replaced, so you see the effect at once. In production keep the default retention of 7 days. Time travel needs the old files, and so can readers and writers that still work on an older version.

In [ ]:
# Task 2a: paste the code from the Workshop Guide here (SQL or Python)

In [ ]:
# -- Validation --
detail_v2 = spark.sql(f"DESCRIBE DETAIL {TABLE_BASE}").first()
history   = spark.sql(f"DESCRIBE HISTORY {TABLE_BASE}").collect()
optimizes = [r for r in history if r["operation"] == "OPTIMIZE"]
assert optimizes, \
    f"OPTIMIZE must be logged in DESCRIBE HISTORY as its own version, got: {[r['operation'] for r in history[:8]]}"
metrics = optimizes[0]["operationMetrics"]
removed, added = int(metrics["numRemovedFiles"]), int(metrics["numAddedFiles"])
assert added < removed, f"OPTIMIZE must have reduced file count: it removed {removed} files and added {added}"
assert optimizes[0]["operationParameters"].get("zOrderBy", "[]") == "[]", \
    "orders_lab is the baseline of Task 3: compact it without ZORDER BY. Re-run the Preparation cell and Task 2a."
print(f"Task 2a OK")
print(f"  Files removed by OPTIMIZE : {removed}")
print(f"  Files added by OPTIMIZE   : {added}")
print(f"  Files now                 : {detail_v2['numFiles']}")

In [ ]:
# Task 2b: paste the code from the Workshop Guide here (SQL or Python)

In [ ]:
# Task 2c: paste the code from the Workshop Guide here (SQL or Python)

In [ ]:
# -- Validation --
rows_after_vacuum = spark.table(TABLE_BASE).count()
all_ops           = [r["operation"] for r in spark.sql(f"DESCRIBE HISTORY {TABLE_BASE}").collect()]
assert rows_after_vacuum > 0, "Table must still contain data after VACUUM"
assert any(op.startswith("VACUUM") for op in all_ops), \
    "VACUUM START/END must appear in DESCRIBE HISTORY"
print(f"Task 2c OK")
print(f"  Rows still present    : {rows_after_vacuum}")
print(f"  History now contains  : OPTIMIZE + VACUUM versions")

## Setup for Task 3: Create Isolated Table Copies

Task 3 compares two layouts, so each layout needs its own copy of the compacted table. `DEEP CLONE` creates a full, independent Delta table: changes to the clone never affect the source.

The next cell is provided. Run it as it is, without changes.

In [ ]:
# PROVIDED (run as it is): make two independent copies of the compacted table
spark.sql(f"CREATE OR REPLACE TABLE {TABLE_ZORDER} DEEP CLONE {TABLE_BASE}")
spark.sql(f"CREATE OR REPLACE TABLE {TABLE_LIQUID} DEEP CLONE {TABLE_BASE}")

print("Deep clone complete:")
print(f"  {TABLE_ZORDER}")
print(f"  {TABLE_LIQUID}")

## Task 3 — Z-ORDER vs Liquid Clustering vs Baseline

The dashboard filters orders by `product_id`. After compaction the rows of one product are still spread over the files, so a filter on one product has to open most of them. Clustering stores rows with similar values in the same files. The min/max statistics of each file then let Delta skip the files that cannot hold the product. Z-ordering is the older way to do this. Databricks recommends liquid clustering (`CLUSTER BY`) for new tables.

The setup cell above made two copies of the compacted table. **Task 3a** Z-orders one copy, and **Task 3b** turns on liquid clustering on the other. Then run the **Benchmark** cell and choose the layout in the Workshop Guide (**Task 3 choice**, in the Polish app **Wybór w Zadaniu 3**). The code for Task 3a and Task 3b is in the Workshop Guide.

In [ ]:
# Task 3a: paste the code from the Workshop Guide here (SQL or Python)

In [ ]:
# Task 3b: paste the code from the Workshop Guide here (SQL or Python)

In [ ]:
# -- Validation --
detail_zorder = spark.sql(f"DESCRIBE DETAIL {TABLE_ZORDER}").first()
detail_liquid = spark.sql(f"DESCRIBE DETAIL {TABLE_LIQUID}").first()
assert detail_zorder["numFiles"] > 0, "Z-ORDER table should still hold data files — re-run Setup for Task 3 and Task 3a"
assert list(detail_liquid["clusteringColumns"]) == ["product_id"], \
    f"DESCRIBE DETAIL clusteringColumns should be ['product_id'], got {detail_liquid['clusteringColumns']}"
zorder_optimizes = [
    r for r in spark.sql(f"DESCRIBE HISTORY {TABLE_ZORDER}").collect()
    if r["operation"] == "OPTIMIZE"
]
assert zorder_optimizes, "OPTIMIZE must appear in the Z-ORDER table history"
assert "product_id" in str(zorder_optimizes[0]["operationParameters"]), \
    f"The OPTIMIZE should record ZORDER BY product_id, got {zorder_optimizes[0]['operationParameters']}"
history_liquid = spark.sql(f"DESCRIBE HISTORY {TABLE_LIQUID}").collect()
cluster_by     = max([r["version"] for r in history_liquid if r["operation"] == "CLUSTER BY"], default=-1)
assert any(r["operation"] == "OPTIMIZE" and r["version"] > cluster_by for r in history_liquid), \
    "Run OPTIMIZE after ALTER TABLE ... CLUSTER BY (Task 3b): only OPTIMIZE clusters the files"
print("Task 3 OK")
print(f"  Z-ORDER : {detail_zorder['numFiles']} files, ZORDER BY recorded in history")
print(f"  Liquid  : clusteringColumns = {list(detail_liquid['clusteringColumns'])}, OPTIMIZE applied")

## Benchmark: Query Performance Comparison

Use a benchmark like this at work when you choose a layout: run the query that users run and compare how many files it needs. All three tables are compacted. The baseline has no clustering, and the other two tables are clustered by `product_id`. The next cell runs the same query for one product on each table and prints these columns:

- `Files` is the total file count of the table. It shows the effect of compaction, not data skipping.
- `Files w/ match` is the number of files that contain `PROD_0042`, counted from the hidden `_metadata.file_path` column. Clustering puts the product into fewer files, so the min/max statistics per file let Delta skip the rest.
- `Time (s)` is the wall-clock time of `WHERE product_id = 'PROD_0042'`. On a table this small the time changes from run to run, so compare the files first.

To see the files that the query actually read and skipped, open the **Query Profile** (serverless: "See performance" under the cell) or **Spark UI → SQL tab** (classic compute).

In [ ]:
# Benchmark: count the files that hold one product and time the query on each table
import time

BENCHMARK_PRODUCT = "PROD_0042"

def time_query(table):
    """Run a point-query for one product and return (elapsed_seconds, row_count)."""
    t0 = time.time()
    result = spark.sql(f"""
        SELECT product_id, COUNT(*) AS order_count, ROUND(SUM(total_amount), 2) AS total_revenue
        FROM {table}
        WHERE product_id = '{BENCHMARK_PRODUCT}'
        GROUP BY product_id
    """).collect()
    elapsed = time.time() - t0
    rows = result[0]["order_count"] if result else 0
    return elapsed, rows

def files_with_matches(table):
    """Files that contain the product — a lower bound on the files Delta has to read."""
    return spark.sql(f"""
        SELECT COUNT(DISTINCT _metadata.file_path) AS n
        FROM {table}
        WHERE product_id = '{BENCHMARK_PRODUCT}'
    """).first()["n"]

print(f"Benchmark: WHERE product_id = '{BENCHMARK_PRODUCT}'")
print(f"{'Strategy':<36} {'Files':>6} {'Files w/ match':>15} {'Time (s)':>9} {'Rows matched':>13}")
print("-" * 83)

for table, label in [
    (TABLE_BASE,   "Baseline (compacted, no clustering)"),
    (TABLE_ZORDER, "Z-ORDER by product_id"),
    (TABLE_LIQUID, "Liquid Clustering by product_id"),
]:
    n_files = spark.sql(f"DESCRIBE DETAIL {table}").first()["numFiles"]
    n_match = files_with_matches(table)
    t, rows = time_query(table)
    print(f"{label:<36} {n_files:>6} {n_match:>15} {t:>9.2f} {rows:>13,}")

print()
print("How to read it: 'Files' only reflects compaction. Data skipping shows as")
print("'Files w/ match' < 'Files' — the files without the product can be skipped via min/max stats.")
print("If a table compacted into 1-2 files, there is little left to skip at this data size.")
print("Confirm files pruned in the Query Profile (serverless) or Spark UI SQL tab (classic).")

## Task 4 — Let the Platform Choose the Clustering Keys (stretch)

> Stretch: optional in class. Do it if you finish early, or after the course. No later task needs it. Run **Cleanup** after it, or after Task 3 if you skip it.

Finance plans more filters on the dashboard, so the best clustering keys can change. With automatic liquid clustering, predictive optimization looks at the queries on the table, chooses the clustering keys and changes them when the queries change. It works on Unity Catalog managed tables and needs predictive optimization.

The cell clones the base table to `orders_lab_auto`, turns on automatic liquid clustering and shows the table properties. The code is in the Workshop Guide (**Task 4**).

> Automatic upgrades turn on automatic liquid clustering for new tables by default (rollout from May 22, 2026). So in some workspaces `orders_lab` and its clones already show automatic clustering in their table properties before this task. A table created earlier keeps its old layout until you turn it on yourself, and the command records your choice in the table history.

> Right after you turn it on, `clusteringColumns` can stay empty. Predictive optimization chooses the keys later, in the background, and only when the table gains from it (enough size and enough queries).

In [ ]:
# Task 4: paste the code from the Workshop Guide here (SQL or Python)

In [ ]:
# -- Validation --
props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {TABLE_AUTO}").collect()}
assert props.get("clusterByAuto") == "true", \
    f"Expected table property clusterByAuto=true — did you run ALTER TABLE ... CLUSTER BY AUTO? Got: {props.get('clusterByAuto')}"
# New tables can get clusterByAuto=true by default (automatic upgrades): look for your ALTER TABLE after the latest clone
history_auto = spark.sql(f"DESCRIBE HISTORY {TABLE_AUTO}").collect()
last_clone   = max([r["version"] for r in history_auto if r["operation"] == "CLONE"], default=-1)
auto_alters  = [r for r in history_auto
                if r["operation"] == "CLUSTER BY" and r["version"] > last_clone
                and (r["operationParameters"] or {}).get("clusterByAuto") == "true"]
assert auto_alters, \
    "No ALTER TABLE ... CLUSTER BY AUTO in the history of orders_lab_auto after its latest CLONE " \
    "(new tables can get clusterByAuto by default). Run Task 4."
detail_auto = spark.sql(f"DESCRIBE DETAIL {TABLE_AUTO}").first()
assert detail_auto["numFiles"] > 0, "Cloned table should contain data files"
print("Task 4 OK: Automatic Liquid Clustering enabled")
print(f"  clusterByAuto      : {props.get('clusterByAuto')}")
print(f"  clusteringColumns  : {list(detail_auto['clusteringColumns'])} "
      "(may be empty until predictive optimization picks columns from query patterns)")

## Cleanup

Run this cell only after **Check your work** has passed for every task. It drops the tables that the checks read.

In [ ]:
# Drop the four lab tables
for table in [TABLE_BASE, TABLE_ZORDER, TABLE_LIQUID, TABLE_AUTO]:
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    print(f"Dropped: {table}")
print("All lab tables cleaned up.")

## Lab Complete

In this lab you measured a fragmented Delta table, compacted it and compared layouts for one filter:

| Task | Skill |
|---|---|
| 1 — Inspect | `DESCRIBE DETAIL`, `DESCRIBE HISTORY`: a baseline from the table metadata |
| 2 — Optimize | `OPTIMIZE` + `VACUUM` (2b–2c stretch): file compaction and storage reclaim, in the production order |
| 3 — Compare | Z-ORDER vs Liquid Clustering vs baseline: files that hold one product |
| 4 — Automate (stretch) | `CLUSTER BY AUTO`: predictive optimization chooses the clustering keys |

For new tables, liquid clustering replaces partitioning and Z-ORDER.
To change the clustering columns without rewriting all data, run `ALTER TABLE ... CLUSTER BY (new_col)`. To let the platform choose them, use `CLUSTER BY AUTO`.

← <a href="$../demo/06_delta_optimization">06 — Delta Optimization</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$../demo/07_lakeflow_pipelines">07 — Lakeflow Pipelines →</a>